In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
spark = SparkSession.builder.appName("SparkCompleteNotes").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:12:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/13 06:12:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
numData = spark.range(0,1000000).withColumn("value",col("id")%1000)

In [7]:

numData_repart = numData.repartition(20)


numData_coal = numData_repart.coalesce(20)


numData_repart.write.mode("overwrite").csv("output/repartitioned_data")
numData_coal.write.mode("overwrite").csv("output/coalesced_data")

26/06/13 06:13:12 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [10]:
optimized_df = numData.filter(col("value")>500).filter(col("id")<5000).select("id","value")

In [11]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#2L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 5000))
   +- *(1) Range (0, 1000000, step=1, splits=2)




In [12]:
import time
start_time = time.time()
count_uncached = optimized_df.count()
end_time=time.time()
print(f"1. Optimized execution | count:{count_uncached} | time:{round(end_time-start_time,4)}sec")

1. Optimized execution | count:2495 | time:0.4124sec


In [13]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [14]:
import time
start_time = time.time()
count_uncached = optimized_df.count()
end_time=time.time()
print(f"2. Optimized execution | count:{count_uncached} | time:{round(end_time-start_time,4)}sec")

2. Optimized execution | count:2495 | time:0.614sec


In [15]:
import time
start_time = time.time()
count_uncached = optimized_df.count()
end_time=time.time()
print(f"3.. Optimized execution | count:{count_uncached} | time:{round(end_time-start_time,4)}sec")

3.. Optimized execution | count:2495 | time:0.1831sec
